# Receituário da IlumA

Isto **não** é um tutorial — o passo a passo está no `Regex101.ipynb` da Aula 04,
com o tutorial do token, os papéis do prompt e os parâmetros de amostragem.

Aqui cada seção é um **padrão pronto para copiar** no seu projeto. Fora a conexão,
nenhuma depende da anterior. Use como página de consulta enquanto trabalha.

| padrão | para quê |
|---|---|
| 1 | conectar |
| **2** | **filtrar com regex antes de chamar o modelo** |
| 3 | obter JSON que não quebra |
| 4 | extrair de um abstract inteiro, com few-shot |
| 5 | rodar em lote sem perder trabalho |
| 6 | estimar o custo antes de disparar |
| 7 | detectar alucinação |
| 8 | medir se funcionou |

## Padrão 1 — Conexão

Três coisas mudam em relação a qualquer exemplo da OpenAI: a `base_url`, o nome do
modelo e o token. A `temperature` não pode descer abaixo de 0,5 nesta instalação.

In [ ]:
import getpass
import json
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

BASE_URL = "https://iluma.cnpem.br:4000/v1"
MODELO = "iluma"
TEMP = 0.5          # piso desta instalação: abaixo disso a chamada trava

cliente = OpenAI(
    base_url=BASE_URL,
    api_key=getpass.getpass("Token da IlumA (sk-...): "),
    timeout=60.0,    # sem isto, uma chamada travada fica pendurada para sempre
    max_retries=1,
)


def perguntar(mensagens, **kw):
    kw.setdefault("temperature", TEMP)
    r = cliente.chat.completions.create(model=MODELO, messages=mensagens, **kw)
    return r.choices[0].message.content


print(perguntar([{"role": "user", "content": "Responda apenas: conexão OK."}]))

## Padrão 2 — Filtrar com regex antes de chamar o modelo

**É o padrão mais importante desta página.**

A tentação é mandar o corpus inteiro para o modelo. Mas a maior parte dos resumos
não tem nada do que você procura — e você paga por cada um deles, em dinheiro e em
tempo.

Um regex varre 7 MB de texto em milissegundos e é de graça. Deixe-o fazer a
triagem: só o que passar no filtro vira uma chamada.

Números medidos neste corpus:

| etapa | abstracts | |
|---|---|---|
| com resumo | 6.228 | 100% |
| citam "yield strength" | 754 | 12,1% |
| …e têm um valor em Pa | 480 | 7,7% |
| …e têm uma fórmula de liga | **310** | **5,0%** |

**20× menos chamadas.** Em tokens de entrada, de ~1,81 milhão para ~94 mil: 5,2%
do custo original.

In [ ]:
dados = pd.concat([pd.read_excel(f"savedrecs{i}.xls") for i in range(1, 9)],
                  ignore_index=True)
abstracts = dados["Abstract"].dropna().astype(str).tolist()

# Três peneiras baratas. Ajuste as suas ao que você procura.
CHAVE = re.compile(r"[Yy]ield [Ss]trength")
VALOR = re.compile(r"\d+(?:\.\d+)?\s?[TGMKk]Pa\b")
LIGA = re.compile(r"\b[A-Z][a-z]?\d*(?:\.\d+)?"
                  r"(?:[A-Z][a-z]?\d*(?:\.\d+)?){4,8}\b")


def vale_a_chamada(texto: str) -> bool:
    """O resumo tem chance de conter o que procuramos?"""
    return bool(CHAVE.search(texto) and VALOR.search(texto) and LIGA.search(texto))


candidatos = [a for a in abstracts if vale_a_chamada(a)]

print(f"{len(abstracts)} resumos → {len(candidatos)} candidatos "
      f"({len(candidatos) / len(abstracts):.1%})")
print(f"chamadas economizadas: {len(abstracts) - len(candidatos)}")

### Uma inversão importante

Na hora de **extrair**, você quer ser exigente: regra estreita, poucos falsos
positivos.

Na hora de **filtrar**, o certo é o contrário — seja **generoso**. As duas falhas
custam coisas muito diferentes:

* um falso positivo no filtro custa **uma chamada** (centavos, e o modelo devolve
  vazio);
* um falso negativo custa **o dado**, e você nunca fica sabendo — o resumo é
  descartado antes de qualquer um olhar para ele.

Por isso o filtro deve aceitar as variantes de grafia. Se o artigo escreveu "yield
stress" ou "sigma_y", a peneira estreita joga fora sem avisar.

In [ ]:
# Filtro generoso: alternativas de grafia, e um só dos sinais já basta para passar.
CHAVE_AMPLA = re.compile(
    r"[Yy]ield\s+(?:strength|stress|point)"
    r"|YS\b"
    r"|[Ee]scoamento",
)

amplos = [a for a in abstracts if CHAVE_AMPLA.search(a)]
estreitos = [a for a in abstracts if CHAVE.search(a)]

print(f"filtro estreito: {len(estreitos)} resumos")
print(f"filtro amplo:    {len(amplos)} resumos "
      f"(+{len(amplos) - len(estreitos)} que o estreito perderia)")

## Padrão 3 — JSON que não quebra

Mesmo pedindo JSON puro, o modelo às vezes embrulha em cercas de markdown ou
escreve uma frase antes. Duas linhas de limpeza resolvem quase tudo; o resto é
não deixar uma resposta ruim derrubar o laço.

In [ ]:
def ler_json(texto: str) -> dict:
    """Tolera cercas de markdown e texto solto em volta do JSON."""
    t = texto.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t)
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        # última tentativa: o maior trecho entre chaves
        m = re.search(r"\{.*\}", t, re.S)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
    print("  (resposta não era JSON):", t[:120])
    return {}

## Padrão 4 — Extrair de um abstract inteiro

Na Aula 04 o modelo lia uma sentença por vez. Lendo o resumo inteiro ele resolve
referências entre frases ("a liga acima") e enxerga a condição do ensaio, que
costuma estar noutro período.

O par `user`/`assistant` do meio é o **few-shot**: nunca aconteceu, foi escrito
por você para mostrar o formato.

In [ ]:
SISTEMA = """Você extrai dados de resumos de artigos sobre ligas metálicas.

Do resumo fornecido, extraia os pares (composição da liga, limite de escoamento).

Regras:
1. Apenas o limite de escoamento (yield strength). Ignore dureza, limite de
   resistência à tração e alongamento.
2. Se o número for uma DIFERENÇA ou COMPARAÇÃO ("500 MPa higher than"), ignore.
3. Registre a condição do ensaio: "tracao", "compressao" ou "nao_informado".
4. Registre a temperatura do ensaio, se houver.
5. Converta para MPa (1 GPa = 1000 MPa).
6. Responda APENAS com JSON válido, sem cercas de markdown.
7. Se não houver nada, responda {"extracoes": []}."""

EXEMPLO_ENTRADA = (
    "A series of TiC/Al0.6CrFe2Ni2 high entropy alloys were prepared by vacuum arc "
    "melting. When TiC content is 2.50 vol%, the yield strength and Vickers hardness "
    "are 576.93 MPa and 244 HV, respectively."
)
EXEMPLO_SAIDA = json.dumps({"extracoes": [{
    "composicao": "Al0.6CrFe2Ni2 + 2.50 vol% TiC",
    "yield_strength_MPa": 576.93,
    "condicao": "nao_informado",
    "temperatura": None,
}]}, ensure_ascii=False)


def extrair(resumo: str) -> list[dict]:
    bruto = perguntar([
        {"role": "system", "content": SISTEMA},
        {"role": "user", "content": EXEMPLO_ENTRADA},
        {"role": "assistant", "content": EXEMPLO_SAIDA},
        {"role": "user", "content": resumo},
    ])
    return ler_json(bruto).get("extracoes", [])

## Padrão 5 — Lote com retomada

Um laço de 310 chamadas leva dezenas de minutos. Se a rede cair no item 200 e você
não tiver salvado nada, perdeu tudo.

A receita: **salvar a cada item** e **guardar o índice**, para poder continuar de
onde parou.

In [ ]:
SAIDA = Path("extracoes.jsonl")


def rodar_lote(textos, comeco=0, limite=None):
    """Processa `textos` a partir de `comeco`, gravando uma linha por item.

    Cada linha do .jsonl tem o índice: relendo o arquivo você sabe exatamente
    onde parou, e roda de novo só o que falta.
    """
    feitos = set()
    if SAIDA.exists():
        with SAIDA.open() as f:
            feitos = {json.loads(l)["i"] for l in f if l.strip()}
        print(f"{len(feitos)} itens já processados — serão pulados")

    alvos = list(enumerate(textos))[comeco:]
    if limite:
        alvos = alvos[:limite]

    with SAIDA.open("a") as f:
        for i, texto in alvos:
            if i in feitos:
                continue
            try:
                linha = {"i": i, "extracoes": extrair(texto)}
            except Exception as e:                      # rede, timeout, cota
                linha = {"i": i, "erro": f"{type(e).__name__}: {e}"}
                print(f"  [{i}] falhou: {type(e).__name__}")
            f.write(json.dumps(linha, ensure_ascii=False) + "\n")
            f.flush()                                   # o segredo: gravar já
            time.sleep(0.2)
    print("fim")


# Comece pequeno. Só aumente depois de ver o custo (padrão 6).
rodar_lote(candidatos, limite=5)

## Padrão 6 — Estimar o custo antes de disparar

Nunca dispare o laço completo sem saber quanto ele custa. Rode uma dúzia, olhe o
consumo real em **Console → Usage Logs** e multiplique.

A conta de guardanapo: **~4 caracteres por token** em inglês (em português, menos —
o mesmo texto gasta mais tokens).

In [ ]:
def estimar(textos, sistema=SISTEMA, exemplo=EXEMPLO_ENTRADA + EXEMPLO_SAIDA):
    """Estimativa grosseira de tokens de ENTRADA do lote inteiro."""
    fixo = len(sistema) + len(exemplo)          # repetido em toda chamada
    chars = sum(len(t) for t in textos) + fixo * len(textos)
    tokens = chars / 4
    return len(textos), tokens


n, tok = estimar(candidatos)
n_tudo, tok_tudo = estimar(abstracts)
print(f"com filtro:  {n:5d} chamadas · ~{tok/1000:8.0f} k tokens de entrada")
print(f"sem filtro:  {n_tudo:5d} chamadas · ~{tok_tudo/1000:8.0f} k tokens de entrada")
print(f"o filtro economiza {100 * (1 - tok / tok_tudo):.1f}% da entrada")
print("\nO número real está em Console → Usage Logs. Rode 12 e confira.")

## Padrão 7 — Detectar alucinação

Um regex nunca devolve um número que não está no texto. Um modelo pode — e devolve
com a mesma confiança do acerto.

A checagem mais barata que existe: **o valor extraído aparece no texto de origem?**
Não pega tudo, mas pega a maior parte.

In [ ]:
def confere_no_texto(extracoes: list[dict], origem: str) -> list[dict]:
    """Marca cada extração conforme o valor apareça, ou não, no texto original.

    Compara o número tanto em MPa quanto em GPa, porque o prompt pede conversão —
    "5.14 GPa" no texto vira 5140.0 na resposta, e ingenuamente pareceria invenção.
    """
    for e in extracoes:
        v = e.get("yield_strength_MPa")
        if v is None:
            e["confere"] = None
            continue
        formas = {f"{v:g}", f"{v / 1000:g}"}
        e["confere"] = any(f in origem for f in formas)
    return extracoes


exemplo = candidatos[0]
print(exemplo[:200], "...\n")
for e in confere_no_texto(extrair(exemplo), exemplo):
    marca = {True: "ok", False: "NÃO ESTÁ NO TEXTO", None: "sem valor"}[e["confere"]]
    print(f"  {e.get('composicao')}: {e.get('yield_strength_MPa')} MPa  → {marca}")

## Padrão 8 — Medir se funcionou

A pergunta não é "rodou?", é "quanto está certo?". Sem esse número você tem uma
impressão; com ele, um resultado — e algo para escrever no relatório.

O placar do regex, para comparar: **45 linhas extraídas, 44 corretas**.

In [ ]:
def amostra_para_conferir(caminho=SAIDA, n=30, semente=42):
    """Sorteia extrações para revisão manual, com a origem ao lado."""
    linhas = []
    with Path(caminho).open() as f:
        for l in f:
            if not l.strip():
                continue
            d = json.loads(l)
            for e in d.get("extracoes", []):
                linhas.append({
                    "i": d["i"],
                    "composicao": e.get("composicao"),
                    "MPa": e.get("yield_strength_MPa"),
                    "condicao": e.get("condicao"),
                    "origem": candidatos[d["i"]][:300],
                })
    tab = pd.DataFrame(linhas)
    print(f"{len(tab)} extrações no arquivo")
    return tab.sample(min(n, len(tab)), random_state=semente)


# Confira cada linha à mão e preencha a coluna. A precisão é a média dela.
# amostra = amostra_para_conferir()
# amostra["correta"] = None
# amostra

## Adaptando ao seu projeto

O que muda de um problema para outro é sempre a mesma tríade:

1. **as peneiras** do padrão 2 — quais palavras indicam que vale a chamada;
2. **o system prompt** do padrão 3 — o que conta como um dado válido, e o que
   ignorar;
3. **o exemplo** few-shot — uma entrada da sua área com a resposta ideal.

O resto (JSON, lote, custo, alucinação, validação) serve igual.

**Exercícios**

1. Troque as peneiras para a sua propriedade e meça a redução de chamadas.
2. Compare o filtro estreito com o amplo: quantos resumos a mais entram, e quantos
   deles o modelo aproveita de fato?
3. Rode 30 com filtro e 30 sem, e compare quantas extrações válidas saem de cada.
4. Meça a precisão pelo padrão 8 e compare com o placar do regex da Aula 04.
5. Force uma alucinação: peça um campo que não existe no texto e veja o que vem.